# 🌧️ Pipeline Imputasi Data Curah Hujan & Reanalisis Firebase
**Proyek**: RainFall Meteorology & Satellite/Reanalysis Imputation Pipeline

## 📌 Tujuan Notebook
Notebook ini memproses data estimasi curah hujan Satelit Oya (`Rainfall_Oya_TimeSeries_UNIX.csv`) dan data Reanalisis ERA5-Land (`ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv`), melakukan kalkulasi variabel meteorologi standar BMKG/AWS, dan memperbarui database Firebase Realtime Database (`/auto_weather_stat/{STATION_ID}/data`).

## 📏 Skema Field Standar Firebase
- `temperature`: Suhu Udara (°C)
- `humidity`: Kelembaban Udara (%)
- `pressure`: Tekanan Udara (hPa)
- `dew`: Titik Embun (°C)
- `wind_speed`: Kecepatan Angin (m/s)
- `rainrate`: Curah Hujan per 30 Menit (mm/30-min)
- `rainfall`: Akumulasi Harian Curah Hujan (mm, reset 07:00 WIB / 00:00 UTC)
- `timestamp`: UNIX UTC Timestamp detik (10-digit integer)

In [ ]:
# 1. IMPOR DEPENDENSI UTAMA & PEMERIKSAAN MODUL
import os
import sys
import glob
import time
import json
import warnings
from datetime import datetime, timezone
import subprocess

warnings.filterwarnings('ignore')

# Auto-install dependensi jika belum tersedia
required_packages = ['firebase-admin', 'pandas', 'numpy', 'matplotlib', 'tqdm']
for pkg in required_packages:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f"[INFO] Installing missing package: {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import firebase_admin
from firebase_admin import credentials, db

print("✅ Dependensi berhasil dimuat!")

In [ ]:
# 2. KONFIGURASI PARAMETER UTAMA
STATION_ID = "id-01"  # Target stasiun AWS
FIREBASE_URL = "https://staklimjerukagung-default-rtdb.asia-southeast1.firebasedatabase.app/"
BATCH_SIZE = 2000     # Jumlah record per batch update Firebase
DRY_RUN = False       # Set True untuk simulasi tanpa mengubah database Firebase

# Filter Rentang Waktu (Format 'YYYY-MM-DD' atau Set None untuk seluruh data)
START_DATE = "2023-01-01"
END_DATE   = "2023-12-31"

# Deteksi Otomatis Path Dataset (Kaggle Cloud vs Local Environment)
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')

if IS_KAGGLE:
    print("[INFO] Running in Kaggle Environment")
    candidates_oya = glob.glob('/kaggle/input/**/Rainfall_Oya_TimeSeries_UNIX.csv', recursive=True)
    PATH_OYA_CSV = candidates_oya[0] if candidates_oya else '/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/Google_Earth_Engine/Data_Satelit/Rainfall_Oya_TimeSeries_UNIX.csv'
    candidates_era5 = glob.glob('/kaggle/input/**/ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv', recursive=True)
    PATH_ERA5_CSV = candidates_era5[0] if candidates_era5 else '/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/Google_Earth_Engine/Data_Satelit/ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv'
else:
    print("[INFO] Running in Local Environment")
    PATH_OYA_CSV = r"d:\Github\Projek_Rainfall\Google_Earth_Engine\Data_Satelit\Rainfall_Oya_TimeSeries_UNIX.csv"
    PATH_ERA5_CSV = r"d:\Github\Projek_Rainfall\Google_Earth_Engine\Data_Satelit\ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv"

print(f"📌 Target Station  : {STATION_ID}")
print(f"📁 Path Oya CSV     : {PATH_OYA_CSV}")
print(f"📁 Path ERA5 CSV    : {PATH_ERA5_CSV}")
print(f"📅 Rentang Waktu    : {START_DATE if START_DATE else 'Awal'} s.d. {END_DATE if END_DATE else 'Akhir'}")
print(f"⚙️ Mode DRY_RUN     : {DRY_RUN}")

In [ ]:
# 3. AUTENTIKASI & INISIALISASI FIREBASE
def initialize_firebase(cred_path=None, database_url=FIREBASE_URL):
    if firebase_admin._apps:
        print("✅ Firebase SDK sudah terinisialisasi.")
        return True
        
    candidates = [
        'D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json',
        'staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json',
        '../staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json'
    ]
    for p in glob.glob('**/*firebase-adminsdk*.json', recursive=True):
        candidates.append(p)
    for p in glob.glob('../*firebase-adminsdk*.json'):
        candidates.append(p)
    if os.path.exists('/kaggle/input'):
        for p in glob.glob('/kaggle/input/**/*firebase-adminsdk*.json', recursive=True):
            candidates.append(p)
            
    target_cert = None
    for c in candidates:
        if os.path.exists(c):
            target_cert = c
            break
            
    if target_cert:
        try:
            cred = credentials.Certificate(target_cert)
            firebase_admin.initialize_app(cred, {'databaseURL': database_url})
            print(f"✅ Firebase berhasil terinisialisasi menggunakan kredensial: `{target_cert}`")
            return True
        except Exception as e:
            print(f"❌ Gagal menginisialisasi Firebase: {e}")
            return False
    else:
        print("⚠️ File kredensial JSON Firebase tidak ditemukan.")
        return False

firebase_ready = initialize_firebase()

In [ ]:
# 4. PEMUATAN DATASET MENTAH
def load_raw_datasets(path_oya, path_era5):
    if not os.path.exists(path_oya):
        raise FileNotFoundError(f"Dataset Oya CSV tidak ditemukan: {path_oya}")
    
    print(f"⏳ Memuat dataset Satelit Oya: {path_oya}")
    df_oya = pd.read_csv(path_oya)
    print(f"   - Total baris Oya mentah: {len(df_oya):,}")
    
    df_era5 = None
    if os.path.exists(path_era5):
        print(f"⏳ Memuat dataset Reanalisis ERA5: {path_era5}")
        df_era5 = pd.read_csv(path_era5)
        print(f"   - Total baris ERA5 mentah: {len(df_era5):,}")
    else:
        print(f"⚠️ Dataset ERA5 tidak ditemukan: {path_era5}")
        
    return df_oya, df_era5

df_oya_raw, df_era5_raw = load_raw_datasets(PATH_OYA_CSV, PATH_ERA5_CSV)

In [ ]:
# 5. PEMBERSIHAN, PEMPROSESAN & PENGGABUNGAN DATA
def process_and_merge_datasets(df_oya, df_era5=None, start_date=START_DATE, end_date=END_DATE):
    # a. Process Satelit Oya
    df_clean_oya = df_oya.dropna(subset=['precipitation_mmhr']).copy()
    df_clean_oya['datetime_utc'] = pd.to_datetime(df_clean_oya['datetime_utc'], utc=True)
    df_clean_oya = df_clean_oya.sort_values('datetime_utc').reset_index(drop=True)
    
    if start_date is not None:
        start_dt = pd.to_datetime(start_date, utc=True)
        df_clean_oya = df_clean_oya[df_clean_oya['datetime_utc'] >= start_dt]
    if end_date is not None:
        end_dt = pd.to_datetime(end_date, utc=True) + pd.Timedelta(days=1)
        df_clean_oya = df_clean_oya[df_clean_oya['datetime_utc'] < end_dt]
        
    df_clean_oya['precipitation_mmhr'] = df_clean_oya['precipitation_mmhr'].clip(lower=0.0)
    df_clean_oya['rainrate'] = df_clean_oya['precipitation_mmhr'].round(4)
    df_clean_oya['date_group'] = df_clean_oya['datetime_utc'].dt.floor('D')
    df_clean_oya['rainfall'] = df_clean_oya.groupby('date_group')['rainrate'].cumsum().round(4)
    df_clean_oya['timestamp'] = df_clean_oya['datetime_utc'].apply(lambda x: int(x.timestamp()))
    df_clean_oya = df_clean_oya.sort_values('timestamp').reset_index(drop=True)
    
    # b. Process Reanalisis ERA5 jika tersedia
    df_clean_era5 = None
    if df_era5 is not None:
        df_era5_work = df_era5.copy()
        df_era5_work['datetime_utc'] = pd.to_datetime(df_era5_work['datetime_utc'], utc=True)
        if start_date is not None:
            df_era5_work = df_era5_work[df_era5_work['datetime_utc'] >= pd.to_datetime(start_date, utc=True)]
        if end_date is not None:
            df_era5_work = df_era5_work[df_era5_work['datetime_utc'] < pd.to_datetime(end_date, utc=True) + pd.Timedelta(days=1)]
            
        # Hitung kecepatan angin dari komponen u & v
        u = df_era5_work['u_component_of_wind_10m_ms']
        v = df_era5_work['v_component_of_wind_10m_ms']
        df_era5_work['wind_speed'] = np.sqrt(u**2 + v**2).round(2)
        df_era5_work['timestamp'] = df_era5_work['datetime_utc'].apply(lambda x: int(x.timestamp()))
        
        df_clean_era5 = df_era5_work[[
            'timestamp',
            'temperature_2m_C',
            'humidity_2m_pct',
            'surface_pressure_hPa',
            'dewpoint_temperature_2m_C',
            'wind_speed'
        ]].rename(columns={
            'temperature_2m_C': 'temperature',
            'humidity_2m_pct': 'humidity',
            'surface_pressure_hPa': 'pressure',
            'dewpoint_temperature_2m_C': 'dew'
        })
        
        for col in ['temperature', 'humidity', 'pressure', 'dew', 'wind_speed']:
            df_clean_era5[col] = df_clean_era5[col].round(2)
            
        df_clean_era5 = df_clean_era5.sort_values('timestamp').reset_index(drop=True)
        
    # c. Merge Oya & ERA5 via merge_asof (tolerance 1 jam)
    if df_clean_era5 is not None:
        era5_match_cols = [c for c in df_clean_era5.columns if c != 'timestamp']
        df_merged = pd.merge_asof(
            df_clean_oya[['timestamp', 'datetime_utc', 'rainrate', 'rainfall']],
            df_clean_era5,
            on='timestamp',
            direction='nearest',
            tolerance=3600
        )
    else:
        df_merged = df_clean_oya[['timestamp', 'datetime_utc', 'rainrate', 'rainfall']].copy()
        
    return df_merged

df_merged = process_and_merge_datasets(df_oya_raw, df_era5_raw, START_DATE, END_DATE)
print(f"✅ Pemrosesan Selesai! Total record terproses: {len(df_merged):,} baris.")
print(f"   - Rentang Waktu: {df_merged['datetime_utc'].min()} s.d {df_merged['datetime_utc'].max()}")
display(df_merged.head(5))

In [ ]:
# 6. RINGKASAN STATISTIK & VISUALISASI DATA HUJAN
def plot_rainfall_summary(df, station_id=STATION_ID):
    plt.figure(figsize=(14, 5))
    plt.plot(df['datetime_utc'], df['rainrate'], color='dodgerblue', alpha=0.7, label='Rainrate (mm/30-min)')
    plt.plot(df['datetime_utc'], df['rainfall'], color='crimson', alpha=0.5, label='Rainfall Kumulatif (mm)')
    plt.title(f'Statistik Curah Hujan — Stasiun {station_id}', fontsize=12, fontweight='bold')
    plt.xlabel('Waktu (UTC)')
    plt.ylabel('Curah Hujan (mm)')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_rainfall_summary(df_merged)

In [ ]:
# 7. MESIN UNGGAH BATCH FIREBASE
def upload_to_firebase(df, station_id=STATION_ID, dry_run=DRY_RUN, batch_size=BATCH_SIZE):
    if dry_run:
        print("⚠️ [DRY_RUN = True] Pengunggahan ke Firebase dilewati. Tidak ada perubahan dilakukan di database.")
        return
        
    if not firebase_admin._apps:
        print("❌ Firebase belum terinisialisasi. Pengunggahan dibatalkan.")
        return
        
    target_path = f'/auto_weather_stat/{station_id}/data'
    print(f"🚀 Memulai pengunggahan {len(df):,} record ke Firebase `{target_path}`...")
    ref_data = db.reference(target_path)
    
    records = df.to_dict(orient='records')
    total_records = len(records)
    valid_fields = ['rainrate', 'rainfall', 'temperature', 'humidity', 'pressure', 'dew', 'wind_speed']
    
    for i in tqdm(range(0, total_records, batch_size), desc=f"Mengunggah Batch ({station_id})"):
        batch = records[i:i + batch_size]
        update_dict = {}
        for rec in batch:
            ts_str = str(int(rec['timestamp']))
            update_dict[f"{ts_str}/timestamp"] = int(rec['timestamp'])
            
            for f_col in valid_fields:
                if f_col in rec and pd.notna(rec[f_col]):
                    update_dict[f"{ts_str}/{f_col}"] = float(rec[f_col])
                    
        try:
            ref_data.update(update_dict)
        except Exception as e:
            print(f"❌ Error saat mengunggah batch indeks {i}: {e}")
            time.sleep(1)
            
    print(f"🎉 Pengunggahan selesai! Total {total_records:,} record berhasil tersimpan di `{target_path}`.")

# Jalankan Pengunggahan ke Firebase
upload_to_firebase(df_merged)

In [ ]:
# 8. VERIFIKASI & INSPEKSI PASCA UNGGAH
def verify_firebase_data(station_id=STATION_ID, sample_count=5):
    if not firebase_admin._apps:
        print("⚠️ Firebase tidak terinisialisasi, verifikasi remote dilewati.")
        return
        
    print(f"🔍 Mengecek {sample_count} record sampel dari Firebase stasiun `{station_id}`...")
    ref = db.reference(f'/auto_weather_stat/{station_id}/data')
    sample = ref.order_by_key().limit_to_last(sample_count).get()
    
    if sample:
        print(json.dumps(sample, indent=2))
        print(f"✅ Data stasiun `{station_id}` di Firebase terverifikasi valid!")
    else:
        print(f"⚠️ Tidak ada data ditemukan pada path `/auto_weather_stat/{station_id}/data`.")

verify_firebase_data()